# PN19 — Two-parent information lock

## TL;DR

A fresh, sealed ARA split predicted **900,000,000,013** from the unused anchor **900,000,000,000** before primality was opened. Independent validation passed **38/38** checks. Phase A alone gave the exact `+13` correction; the A∩B information lock made the result definitive. A post-target 1,000-anchor audit found Phase A exact **93.2%** of the time versus **28.0%** for the p29-wheel control. The exact method remains a two-mask decomposition of an established segmented sieve.

## Context & Methods

**Question.** Can all lower prime children be folded into two complete ARA parent waves, with their relation acting as an Information³ lock?

**Frozen split.** Generate every prime child through `floor(sqrt(2N))`. Split the ordered children at the cumulative-log-weight midpoint. Phase A contains the smaller frequent gates; Phase B contains the larger sparse gates. A candidate survives only when both masks are one.

**Key assumptions and controls.** The target and scripts were hash-frozen. The primary did not contain a target primality function. The A∩B result must equal a complete segmented sieve; exactness is therefore a crosswalk result, not a new primality theorem. The 1,000-anchor robustness run is explicitly post-target and exploratory.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

HERE = Path.cwd()
prediction = json.loads((HERE / 'PN19_TWO_PARENT_INFORMATION_LOCK_PREDICTION.json').read_text())
validation = json.loads((HERE / 'PN19_TWO_PARENT_INFORMATION_LOCK_VALIDATION.json').read_text())
robustness = json.loads((HERE / 'PN19_POST_TARGET_SECOND_GO_ROBUSTNESS.json').read_text())
target = prediction['target']
assert validation['all_passed'] and validation['candidate_is_first_prime_above_anchor']
print('Loaded sealed prediction, independent validation, and exploratory robustness data.')

Loaded sealed prediction, independent validation, and exploratory robustness data.


## Data

In [2]:
target_summary = pd.DataFrame([{
    'anchor': target['anchor'],
    'Phase A first': target['phase_a_first_survivor_offset'],
    'Phase B first': target['phase_b_first_survivor_offset'],
    'A∩B lock': target['information_lock_offset'],
    'sealed candidate': target['predicted_integer'],
    'E_A': target['split']['teara_phase_a'],
    'E_B': target['split']['teara_phase_b'],
    'children': target['child_count'],
}])
target_summary

         anchor  Phase A first  Phase B first  ...       E_A       E_B  children
0  900000000000             13              1  ...  0.999994  1.000006    102973

[1 rows x 8 columns]

In [3]:
development = pd.DataFrame(prediction['development'])[[
    'anchor', 'phase_a_first_survivor_offset', 'phase_b_first_survivor_offset',
    'information_lock_offset', 'either_parent_is_second_go_success'
]]
by_scale = pd.DataFrame(robustness['by_scale'])
development

         anchor  ...  either_parent_is_second_go_success
0     100000000  ...                                True
1    1000000000  ...                                True
2   10000000000  ...                                True
3  100000000000  ...                                True
4  400000000000  ...                                True
5  700000000000  ...                                True

[6 rows x 5 columns]

## Results

In [4]:
phase_a = np.fromfile(HERE / target['phase_a_mask_file'], dtype=np.uint8)
phase_b = np.fromfile(HERE / target['phase_b_mask_file'], dtype=np.uint8)
lock = np.fromfile(HERE / target['information_lock_mask_file'], dtype=np.uint8)

canvas = Image.new('RGB', (1400, 780), 'white')
draw = ImageDraw.Draw(canvas)
draw.text((30, 20), 'PN19 two-parent information lock', fill='#111827')
draw.text((30, 45), 'Fresh target masks: yellow survives, blue collides', fill='#374151')
left, top, cell_w, cell_h = 165, 90, 18, 55
for row_index, (name, values) in enumerate([('Phase A', phase_a), ('Phase B', phase_b), ('A AND B', lock)]):
    y = top + row_index * (cell_h + 18)
    draw.text((30, y + 18), name, fill='#111827')
    for offset in range(1, 65):
        x = left + (offset - 1) * cell_w
        colour = '#f4c95d' if values[offset] else '#3f6c9e'
        draw.rectangle((x, y, x + cell_w - 2, y + cell_h), fill=colour)
        if offset % 4 == 1:
            draw.text((x, y + cell_h + 2), str(offset), fill='#4b5563')
lock_x = left + (target['information_lock_offset'] - 1) * cell_w
draw.line((lock_x, top - 8, lock_x, top + 3*(cell_h+18)-15), fill='#dc2626', width=3)
draw.text((lock_x + 5, top - 8), 'first lock +13', fill='#dc2626')

chart_left, chart_top, chart_right, chart_bottom = 120, 410, 1340, 735
draw.text((30, 370), 'Post-target second-go robustness (200 anchors per scale)', fill='#111827')
draw.line((chart_left, chart_top, chart_left, chart_bottom), fill='#111827', width=2)
draw.line((chart_left, chart_bottom, chart_right, chart_bottom), fill='#111827', width=2)
for percent in range(0, 101, 20):
    y = chart_bottom - (chart_bottom-chart_top)*percent/100
    draw.line((chart_left, y, chart_right, y), fill='#e5e7eb')
    draw.text((70, y-7), f'{percent}%', fill='#4b5563')
xs = np.linspace(chart_left+40, chart_right-40, len(by_scale))
series = [('Phase A', 'phase_a_success_rate', '#b45309'), ('Phase B', 'phase_b_success_rate', '#2563eb'), ('p29 control', 'p29_success_rate', '#6b7280')]
for series_index, (label, column, colour) in enumerate(series):
    points = []
    for x, value in zip(xs, by_scale[column]):
        y = chart_bottom - (chart_bottom-chart_top)*float(value)
        points.append((float(x), float(y)))
    draw.line(points, fill=colour, width=4)
    for x, y in points:
        draw.ellipse((x-5, y-5, x+5, y+5), fill=colour)
    draw.text((chart_left + 250*series_index, chart_top-24), label, fill=colour)
for x, scale in zip(xs, by_scale['scale']):
    draw.text((x-18, chart_bottom+8), f"10^{int(np.log10(scale))}", fill='#4b5563')
figure_path = HERE / 'PN19_TWO_PARENT_INFORMATION_LOCK_FIGURE.png'
canvas.save(figure_path)
figure_path.name

'PN19_TWO_PARENT_INFORMATION_LOCK_FIGURE.png'

In [5]:
metrics = pd.DataFrame({
    'measure': ['TE-ARA share', 'survivor density', 'first survivor'],
    'Phase A': [target['split']['teara_phase_a'], target['phase_a_survivor_density'], target['phase_a_first_survivor_offset']],
    'Phase B': [target['split']['teara_phase_b'], target['phase_b_survivor_density'], target['phase_b_first_survivor_offset']],
    'A∩B': [2.0, target['joint_survivor_density'], target['information_lock_offset']],
})
metrics

            measure    Phase A   Phase B        A∩B
0      TE-ARA share   0.999994  1.000006   2.000000
1  survivor density   0.038818  0.950150   0.037018
2    first survivor  13.000000  1.000000  13.000000

## Takeaways

1. The q-free two-parent lock recovered the first prime at a fresh 900-billion anchor.
2. `E_A≈E_B≈1` but local action is highly asymmetric: Phase A survivor density is about 3.88%, while Phase B survivor density is about 95.01%.
3. Phase A's first survivor was exact on 93.2% of the post-target grid, making the proposed second-go behavior quantitatively real.
4. Exactness still requires the relation because Phase A occasionally admits composites made from two larger children.
5. The method retains all lower-child information inside two masks and is mathematically equivalent to an established segmented sieve. It is a useful ARA decomposition, not yet information or speed compression.

## Reproducibility & QA

In [6]:
assert len(phase_a) == len(phase_b) == len(lock) == 65_536
assert np.array_equal(lock, phase_a & phase_b)
assert abs(target['split']['teara_total'] - 2.0) < 1e-15
assert target['information_lock_offset'] == 13
assert robustness['overall']['anchor_count'] == 1000
assert robustness['overall']['phase_a_success_rate'] == 0.932
assert validation['passed_count'] == validation['check_count'] == 38
qa = {
    'sealed_candidate': validation['candidate'],
    'first_prime_confirmed': validation['candidate_is_first_prime_above_anchor'],
    'independent_checks': f"{validation['passed_count']}/{validation['check_count']}",
    'mask_identity': 'A AND B equals stored lock',
    'post_target_anchor_count': robustness['overall']['anchor_count'],
}
qa

{'sealed_candidate': 900000000013, 'first_prime_confirmed': True, 'independent_checks': '38/38', 'mask_identity': 'A AND B equals stored lock', 'post_target_anchor_count': 1000}